# LexFinder KZ — финальная модель (подготовка к inference)

**Что делает ноутбук (задание 1):** берёт лучшую модель из экспериментов (гибрид BM25 + BGE-M3 dense, `w = 0.6`), строит и сохраняет индекс, загружает его как для эксплуатации, проверяет обработку новых и некорректных входов, фиксирует версию и замеряет качество и задержку (`performance.json` для слайда Performance).

**Перед запуском:** Runtime → GPU (T4). Загрузите в `/content`: архив `lexfinder_final.zip` и файл `lexfinder_labour_code.csv`. Затем Runtime → Run all.

**Почему эта модель:** см. `MODEL_CARD.md` (таблица результатов и обоснование).

In [ ]:
!unzip -q -o lexfinder_final.zip -d /content
%cd /content/lexfinder_final
!pip -q install -r requirements-dev.txt
!mkdir -p data && cp /content/lexfinder_labour_code.csv data/

## 1. Построение и сохранение индекса (один раз)

In [ ]:
import json, sys
sys.path.insert(0, '.')
from lexfinder.engine import build_index
meta = build_index('data/lexfinder_labour_code.csv', 'index')
print(json.dumps(meta, ensure_ascii=False, indent=2))

## 2. Загрузка готовой модели для inference, версия, описание входа и выхода

In [ ]:
from lexfinder import LexFinder, MODEL_NAME, MODEL_VERSION
model = LexFinder.load('data/lexfinder_labour_code.csv', 'index', dense='on')   # 'on': упасть, если нейросеть не загрузилась
print(MODEL_NAME, MODEL_VERSION, '| режим:', model.mode)
print(json.dumps(model.info(), ensure_ascii=False, indent=2)[:1500])

In [ ]:
# Вход:  query (строка, 3–500 символов, с буквами/цифрами), top_k (1–20)
# Выход: список статей {rank, article_number, title, chapter, score, snippet} + mode, model_version, no_lexical_match, latency_ms
import pandas as pd
for q in ['Как оплачивается работа ночью?', 'Можно ли отозвать работника из отпуска?', 'Что такое дистанционная работа?']:
    r = model.search(q, top_k=3)
    print(q, '->', [(h['article_number'], h['title'][:40]) for h in r['results']], f"{r['latency_ms']} мс")

## 3. Обработка новых и некорректных входов

In [ ]:
from lexfinder import QueryError
tests = ['', '   ', '??!!', 'я' * 501, 'квантовая хромодинамика', '  Как   уволиться \n по своей инициативе?  ']
for t in tests:
    try:
        r = model.search(t, 3)
        print(repr(t[:40]), '-> OK, no_lexical_match =', r['no_lexical_match'], '| top1 =', r['results'][0]['article_number'])
    except QueryError as e:
        print(repr(t[:40]), '-> отклонён:', e)

## 4. Качество и задержка (→ слайд Performance)

In [ ]:
!python scripts/benchmark.py --data data/lexfinder_labour_code.csv --index index --dense on --repeats 5

## 5. Автотесты (с реальными данными включается регрессионный тест)

In [ ]:
!pytest -q

## 6. Запуск API и проверка запросом (демо для слайда Demonstration)

In [ ]:
import subprocess, time, requests, os
env = dict(os.environ, LEXFINDER_DENSE='on')
srv = subprocess.Popen(['uvicorn', 'lexfinder.api:create_app', '--factory', '--port', '8000'], env=env,
                       stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
for _ in range(120):
    try:
        if requests.get('http://localhost:8000/health', timeout=2).ok: break
    except Exception: time.sleep(2)
print(requests.get('http://localhost:8000/health').json())
print(requests.post('http://localhost:8000/search', json={'query': 'Как оплачиваются командировки?', 'top_k': 3}).json())
print(requests.post('http://localhost:8000/search', json={'query': ''}).status_code)
print(requests.get('http://localhost:8000/metrics').json())

In [ ]:
srv.terminate()
!zip -qr /content/lexfinder_artifacts.zip index performance.json logs
from google.colab import files
files.download('/content/lexfinder_artifacts.zip')   # индекс нужен для Docker/развёртывания